# Лабораторная 6 - Классификация изображений

ФИО, группа:

Классы, происхождение изображений и условия съемки:

## 1. Библиотеки и настройки

При отсутствии библиотек: `%pip install torch pillow matplotlib numpy` отдельной ячейкой

Для каждого из 2-4 классов требуется не менее 30 разных изображений. Папка dataset содержит отдельную папку каждого класса. Например, dataset/mugs и dataset/bottles

В Colab можно загрузить архив dataset.zip с этой структурой через Files. Ячейка ниже распакует его, если папки dataset еще нет

In [ ]:
from pathlib import Path
import hashlib
import json
import random
import zipfile

import numpy as np
import matplotlib.pyplot as plt
from PIL import Image, ImageOps
import torch
from torch import nn
from torch.utils.data import Dataset, DataLoader

plt.rcParams.update({"axes.unicode_minus": False, "figure.figsize": (10, 5)})
DATA_DIR = Path("dataset")
ARCHIVE_PATH = Path("dataset.zip")
OUTPUT_DIR = Path("outputs")
OUTPUT_DIR.mkdir(exist_ok=True)

IMAGE_SIZE = 128
BATCH_SIZE = 16
EPOCHS = 15
LEARNING_RATE = 0.001
SEED = 42
# Включается только для классов, у которых отражение не меняет смысл изображения
TRAIN_HORIZONTAL_FLIP = False

def set_seed(seed):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)

set_seed(SEED)
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Устройство:", device)

In [ ]:
if not DATA_DIR.exists() and ARCHIVE_PATH.is_file():
    destination = Path.cwd().resolve()
    with zipfile.ZipFile(ARCHIVE_PATH) as archive:
        members = []
        for member in archive.infolist():
            target = (destination / member.filename).resolve()
            if not target.is_relative_to(destination):
                raise ValueError("В архиве обнаружен путь за пределами текущей папки")
            # Из архива извлекается только папка dataset
            if Path(member.filename).parts[:1] == (DATA_DIR.name,):
                members.append(member)
        if not members:
            raise ValueError("Архив должен содержать папку dataset с папками классов")
        archive.extractall(destination, members=members)
if not DATA_DIR.is_dir():
    raise FileNotFoundError("Нужна папка dataset с отдельными папками классов")

## 2. Проверка данных

Название папки становится названием класса. Проверяются чтение изображений и одинаковое содержимое пикселей. Похожие кадры одной серии эта проверка не обнаруживает. Их нужно исключить из набора заранее, чтобы почти одинаковые изображения не попали одновременно в обучение и проверку

In [ ]:
EXTENSIONS = {".jpg", ".jpeg", ".png", ".webp"}
class_dirs = sorted(p for p in DATA_DIR.iterdir() if p.is_dir() and not p.name.startswith("."))
class_names = [p.name for p in class_dirs]
if not 2 <= len(class_names) <= 4:
    raise ValueError("Нужно от 2 до 4 папок классов")
class_to_idx = {name: index for index, name in enumerate(class_names)}


def load_rgb(path):
    with Image.open(path) as image:
        return ImageOps.exif_transpose(image).convert("RGB")


records = []
seen = {}
problems = []
for label, folder in enumerate(class_dirs):
    paths = sorted(p for p in folder.rglob("*") if p.is_file() and p.suffix.lower() in EXTENSIONS)
    if len(paths) < 30:
        problems.append(f"{folder.name}: нужно не менее 30 изображений, найдено {len(paths)}")
    print(folder.name, "-", len(paths), "изображений, номер класса:", label)
    for path in paths:
        try:
            rgb = load_rgb(path)
            digest = hashlib.sha256(str(rgb.size).encode() + rgb.tobytes()).hexdigest()
        except Exception as error:
            problems.append(f"Не читается {path}: {error}")
            continue
        if digest in seen:
            problems.append(f"Повтор изображения: {path} и {seen[digest]}")
        else:
            seen[digest] = path
        records.append({"path": path, "label": label})
if problems:
    raise ValueError("Исправьте данные перед продолжением:\n" + "\n".join(problems))
print("Соответствие классов номерам:", class_to_idx)

## 3. Разделение данных

Внутри каждого класса примерно 70% изображений попадает в train, 15% в validation и остаток в test. Seed фиксирует случайное перемешивание при неизменном наборе файлов

Train используется для изменения весов. Validation - для выбора версии модели. Test остается для итоговой оценки

In [ ]:
rng = np.random.default_rng(SEED)
splits = {"train": [], "validation": [], "test": []}
for label, name in enumerate(class_names):
    items = [record for record in records if record["label"] == label]
    order = rng.permutation(len(items))
    shuffled = [items[index] for index in order]
    n_train = int(len(items) * 0.70)
    n_val = int(len(items) * 0.15)
    splits["train"].extend(shuffled[:n_train])
    splits["validation"].extend(shuffled[n_train:n_train + n_val])
    splits["test"].extend(shuffled[n_train + n_val:])
    print(name, "train:", n_train, "validation:", n_val, "test:", len(items) - n_train - n_val)

split_document = {
    "seed": SEED,
    "class_names": class_names,
    "splits": {
        name: [{"file": str(item["path"].relative_to(DATA_DIR)), "label": item["label"]}
               for item in items]
        for name, items in splits.items()
    },
}
(OUTPUT_DIR / "split.json").write_text(json.dumps(split_document, ensure_ascii=False, indent=2), encoding="utf-8")

## 4. Изображения и тензоры

Изображение переводится в RGB и вписывается в квадрат без растяжения. Свободные поля заполняются серым цветом. Числа делятся на 255, оси переставляются из HWC в CHW: каналы, высота, ширина

Dataset возвращает изображение и номер класса, DataLoader собирает их в пакеты. У пакета появляется первая ось N - число изображений

In [ ]:
def prepare_image(image, image_size):
    image = ImageOps.pad(image.convert("RGB"), (image_size, image_size),
                         method=Image.Resampling.BILINEAR, color=(128, 128, 128))
    array = np.asarray(image, dtype=np.float32) / 255.0
    return torch.from_numpy(array.transpose(2, 0, 1).copy())


class ImageDataset(Dataset):
    def __init__(self, records, image_size, horizontal_flip=False):
        self.records = records
        self.image_size = image_size
        self.horizontal_flip = horizontal_flip

    def __len__(self):
        return len(self.records)

    def __getitem__(self, index):
        record = self.records[index]
        image = load_rgb(record["path"])
        if self.horizontal_flip and random.random() < 0.5:
            image = ImageOps.mirror(image)
        return prepare_image(image, self.image_size), record["label"]


train_dataset = ImageDataset(splits["train"], IMAGE_SIZE, TRAIN_HORIZONTAL_FLIP)
val_dataset = ImageDataset(splits["validation"], IMAGE_SIZE)
test_dataset = ImageDataset(splits["test"], IMAGE_SIZE)
train_loader = DataLoader(train_dataset, batch_size=BATCH_SIZE, shuffle=True, num_workers=0,
                          generator=torch.Generator().manual_seed(SEED))
val_loader = DataLoader(val_dataset, batch_size=BATCH_SIZE, shuffle=False, num_workers=0)
test_loader = DataLoader(test_dataset, batch_size=BATCH_SIZE, shuffle=False, num_workers=0)

images, labels = next(iter(train_loader))
print("Форма пакета NCHW:", tuple(images.shape))
print("Тип:", images.dtype, "Диапазон:", images.min().item(), images.max().item())
print("Номера классов:", labels.tolist())
fig, axes = plt.subplots(2, 4, figsize=(12, 6))
for index, ax in enumerate(axes.flat):
    if index < len(images):
        ax.imshow(images[index].permute(1, 2, 0).numpy())
        ax.set_title(class_names[labels[index].item()])
    ax.set_axis_off()
plt.tight_layout()
plt.show()

Наблюдения: признаки, по которым различаются классы, разнообразие фона и ракурсов, влияние изменения размера

## 5. Модель

Сверточные слои выделяют признаки, MaxPool уменьшает размер карт признаков. Последний Linear выдает по одному числу для каждого класса. Это логиты, Softmax перед CrossEntropyLoss не нужен

Модель уже задана. Для первого запуска достаточно исходных настроек

In [ ]:
class SmallCNN(nn.Module):
    def __init__(self, num_classes):
        super().__init__()
        self.features = nn.Sequential(
            nn.Conv2d(3, 16, kernel_size=3, padding=1),
            nn.ReLU(),
            nn.MaxPool2d(2),
            nn.Conv2d(16, 32, kernel_size=3, padding=1),
            nn.ReLU(),
            nn.MaxPool2d(2),
            nn.Conv2d(32, 64, kernel_size=3, padding=1),
            nn.ReLU(),
            nn.MaxPool2d(2),
            nn.AdaptiveAvgPool2d((4, 4)),
        )
        self.classifier = nn.Sequential(
            nn.Flatten(),
            nn.Linear(64 * 4 * 4, 64),
            nn.ReLU(),
            nn.Dropout(0.5),
            nn.Linear(64, num_classes),
        )

    def forward(self, x):
        return self.classifier(self.features(x))


set_seed(SEED)
model = SmallCNN(len(class_names)).to(device)
criterion = nn.CrossEntropyLoss()
optimizer = torch.optim.Adam(model.parameters(), lr=LEARNING_RATE)
print(model)
print("Обучаемых параметров:", sum(p.numel() for p in model.parameters() if p.requires_grad))

## 6. Обучение и выбор модели

На каждом пакете вычисляется ошибка, backward находит градиенты, optimizer.step изменяет веса. Одна эпоха - полный проход по train

После эпохи оценивается validation. Сохраняется копия весов с минимальной ошибкой на validation. Test на этом этапе не используется

Для нового независимого эксперимента ячейки выполняются с начала. Повторный запуск только следующей ячейки продолжит обучение текущей модели

In [ ]:
def evaluate(model, loader):
    model.eval()
    total_loss, correct, total = 0.0, 0, 0
    with torch.no_grad():
        for images, labels in loader:
            images, labels = images.to(device), labels.to(device)
            logits = model(images)
            loss = criterion(logits, labels)
            total_loss += loss.item() * len(labels)
            correct += (logits.argmax(dim=1) == labels).sum().item()
            total += len(labels)
    return total_loss / total, correct / total


history = {"train_loss": [], "val_loss": [], "train_accuracy": [], "val_accuracy": []}
best_val_loss = float("inf")
best_state = None
best_epoch = None
if EPOCHS < 1:
    raise ValueError("Нужна хотя бы одна эпоха")
for epoch in range(1, EPOCHS + 1):
    model.train()
    total_loss, correct, total = 0.0, 0, 0
    for images, labels in train_loader:
        images, labels = images.to(device), labels.to(device)
        optimizer.zero_grad()
        logits = model(images)
        loss = criterion(logits, labels)
        loss.backward()
        optimizer.step()
        total_loss += loss.item() * len(labels)
        correct += (logits.argmax(dim=1) == labels).sum().item()
        total += len(labels)
    train_loss, train_accuracy = total_loss / total, correct / total
    val_loss, val_accuracy = evaluate(model, val_loader)
    history["train_loss"].append(train_loss)
    history["val_loss"].append(val_loss)
    history["train_accuracy"].append(train_accuracy)
    history["val_accuracy"].append(val_accuracy)
    if val_loss < best_val_loss:
        best_val_loss = val_loss
        best_epoch = epoch
        best_state = {name: value.detach().cpu().clone() for name, value in model.state_dict().items()}
    print(f"Эпоха {epoch:2d}/{EPOCHS}: train loss={train_loss:.4f}, accuracy={train_accuracy:.3f} | "
          f"validation loss={val_loss:.4f}, accuracy={val_accuracy:.3f}")
if best_state is None:
    raise RuntimeError("Не удалось получить модель с конечной ошибкой validation")
model.load_state_dict(best_state)
model.eval()
print("Выбрана эпоха:", best_epoch)

In [ ]:
epoch_numbers = np.arange(1, len(history["train_loss"]) + 1)
fig, axes = plt.subplots(1, 2, figsize=(13, 4))
for ax, metric, title in zip(axes, ["loss", "accuracy"], ["Ошибка", "Доля правильных ответов"]):
    ax.plot(epoch_numbers, history[f"train_{metric}"], label="train")
    ax.plot(epoch_numbers, history[f"val_{metric}"], label="validation")
    ax.axvline(best_epoch, color="gray", linestyle="--", label="Выбранная эпоха")
    ax.set(xlabel="Эпоха", ylabel=metric, title=title)
    ax.legend()
    ax.grid(alpha=0.3)
plt.show()

Train на графике измеряется во время обновления весов, с включенным Dropout и возможным отражением изображений. Validation измеряется после эпохи в режиме eval. Поэтому значения не обязаны совпадать даже при близких наборах данных

Наблюдения: изменение ошибки, выбранная эпоха, признаки переобучения или недостаточного обучения

## 7. Итоговая оценка

После выбора настроек оценивается test. Для сравнения используется постоянный прогноз наиболее частого класса train. Настройки по результатам test не подбираются

В матрице ошибок строки соответствуют правильным классам, столбцы - предсказанным

In [ ]:
model.eval()
y_true, y_pred, confidence = [], [], []
test_loss_sum = 0.0
with torch.no_grad():
    for images, labels in test_loader:
        images, labels = images.to(device), labels.to(device)
        logits = model(images)
        test_loss_sum += criterion(logits, labels).item() * len(labels)
        scores, predicted = logits.softmax(dim=1).max(dim=1)
        y_true.extend(labels.cpu().tolist())
        y_pred.extend(predicted.cpu().tolist())
        confidence.extend(scores.cpu().tolist())
y_true, y_pred, confidence = np.array(y_true), np.array(y_pred), np.array(confidence)
test_loss = test_loss_sum / len(y_true)
test_accuracy = np.mean(y_true == y_pred)
train_counts = np.bincount([item["label"] for item in splits["train"]], minlength=len(class_names))
majority_class = int(train_counts.argmax())
baseline_accuracy = np.mean(y_true == majority_class)
print(f"Test: {len(y_true)} изображений, loss={test_loss:.4f}, accuracy={test_accuracy:.3f}")
print(f"Постоянный ответ '{class_names[majority_class]}': accuracy={baseline_accuracy:.3f}")

confusion = np.zeros((len(class_names), len(class_names)), dtype=int)
np.add.at(confusion, (y_true, y_pred), 1)
fig, ax = plt.subplots(figsize=(7, 6))
heatmap = ax.imshow(confusion, cmap="Blues", vmin=0)
fig.colorbar(heatmap, ax=ax, label="Число изображений")
ax.set(xticks=np.arange(len(class_names)), yticks=np.arange(len(class_names)),
       xticklabels=class_names, yticklabels=class_names,
       xlabel="Предсказанный класс", ylabel="Правильный класс", title="Матрица ошибок")
for i in range(len(class_names)):
    for j in range(len(class_names)):
        ax.text(j, i, str(confusion[i, j]), ha="center", va="center",
                color="white" if confusion[i, j] > confusion.max() / 2 else "black")
plt.tight_layout()
plt.show()

In [ ]:
wrong_indices = np.flatnonzero(y_true != y_pred)
selected_indices = wrong_indices[:9] if len(wrong_indices) else np.arange(min(9, len(y_true)))
print("Ошибок:", len(wrong_indices))
if len(wrong_indices) == 0:
    print("Ошибок на этом test нет. Ниже показаны правильные ответы")
n_columns = min(3, len(selected_indices))
n_rows = int(np.ceil(len(selected_indices) / n_columns))
fig, axes = plt.subplots(n_rows, n_columns, figsize=(4.5 * n_columns, 4 * n_rows), squeeze=False)
for ax in axes.flat:
    ax.set_axis_off()
for ax, index in zip(axes.flat, selected_indices):
    image, _ = test_dataset[int(index)]
    ax.imshow(image.permute(1, 2, 0).numpy())
    ax.set_title(f"Файл: {splits['test'][index]['path'].name}\n"
                 f"Верно: {class_names[y_true[index]]}\n"
                 f"Прогноз: {class_names[y_pred[index]]}, оценка {confidence[index]:.2f}", fontsize=9)
plt.tight_layout()
plt.show()

Сравнение модели с постоянным прогнозом:

Какие классы путаются чаще:

Разбор минимум трех ошибочных изображений, если они есть. Если ошибок меньше, разбираются все. При отсутствии ошибок - ограничения маленького test и условия, в которых возможны ошибки

Оценка Softmax не гарантирует правильность ответа и не является измеренной вероятностью ошибки на новых данных

## 8. Сохранение и загрузка

Файл хранит веса, порядок классов и параметры обработки. Для загрузки используется та же архитектура SmallCNN. Ниже сравниваются выходы исходной и восстановленной модели на одном пакете

In [ ]:
MODEL_PATH = OUTPUT_DIR / "classifier.pt"
checkpoint = {
    "state_dict": {name: value.detach().cpu().clone() for name, value in model.state_dict().items()},
    "class_names": class_names,
    "class_to_idx": class_to_idx,
    "image_size": IMAGE_SIZE,
    "resize_mode": "pad_bilinear_gray128",
    "normalization_scale": 255.0,
    "seed": SEED,
    "best_epoch": best_epoch,
}
torch.save(checkpoint, MODEL_PATH)
loaded = torch.load(MODEL_PATH, map_location="cpu", weights_only=True)
loaded_class_names = loaded["class_names"]
restored_model = SmallCNN(len(loaded_class_names)).to(device)
restored_model.load_state_dict(loaded["state_dict"])
restored_model.eval()
model.eval()
check_images, _ = next(iter(test_loader))
with torch.no_grad():
    original_logits = model(check_images.to(device))
    restored_logits = restored_model(check_images.to(device))
torch.testing.assert_close(original_logits, restored_logits)
print("Выходы совпадают. Модель сохранена:", MODEL_PATH)

## 9. Дополнительное изображение

Нужен новый пример одного из заданных классов, который не участвовал в train, validation или test. Путь задается ниже

In [ ]:
NEW_IMAGE_PATH = Path("mydata/new_image.jpg")
if not NEW_IMAGE_PATH.is_file():
    print("Для дополнительного прогноза нужно добавить изображение и указать NEW_IMAGE_PATH")
else:
    new_image = load_rgb(NEW_IMAGE_PATH)
    new_tensor = prepare_image(new_image, loaded["image_size"]).unsqueeze(0).to(device)
    with torch.no_grad():
        probabilities = restored_model(new_tensor).softmax(dim=1)[0].cpu().numpy()
    predicted_class = int(probabilities.argmax())
    fig, ax = plt.subplots()
    ax.imshow(new_image)
    ax.set_title(f"Прогноз: {loaded_class_names[predicted_class]}, оценка {probabilities[predicted_class]:.2f}")
    ax.set_axis_off()
    plt.show()
    for name, value in zip(loaded_class_names, probabilities):
        print(f"{name}: {value:.3f}")

Правильный класс дополнительного изображения и сравнение с прогнозом:

## Выводы

Состав и ограничения набора данных:

Результат обучения и проверки, возможные причины ошибок:

Какие изменения данных или модели стоит проверить в следующем эксперименте:

Для сдачи: выполненный ipynb с графиками, результатами и выводами. Файлы classifier.pt и split.json из outputs сохраняются для повторного запуска. В Colab результаты скачиваются до завершения сеанса